# Model Segmentasi Gigi (untuk Deployment)

Melatih model **segmentasi gigi** yang bisa memprediksi mask pada **foto baru** — inilah
yang dipakai di iPhone untuk mereproduksi input `teeth_masked_crop`. (Roboflow/SAM hanya
melabeli foto lama; ini melatih model sungguhan.)

Alur:
1. **Konversi** mask COCO (`data/masks`, dari notebook 11) → format **YOLO-seg**.
2. Rakit dataset `data/seg_yolo/{train,valid,test}` + `data.yaml`.
3. Latih **YOLOv8-seg** (ultralytics).
4. Evaluasi (mask mAP) + pratinjau prediksi.
5. **Ekspor Core ML** untuk on-device.

Butuh `pycocotools`, `opencv-python`, `ultralytics` (dipasang otomatis bila belum ada).

In [1]:
import os, glob, json, shutil, sys, subprocess
import numpy as np
def _ensure(pip_name, import_name):
    try: __import__(import_name)
    except ImportError: subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pip_name])
_ensure('pycocotools', 'pycocotools'); _ensure('opencv-python-headless', 'cv2')
import cv2
from pycocotools import mask as coco_mask

try:    _HERE = os.path.dirname(os.path.abspath(__file__))
except NameError: _HERE = os.getcwd()
PROJECT_ROOT = os.path.dirname(_HERE) if os.path.basename(_HERE) == 'notebooks' else _HERE
DATA_MASKS = os.path.join(PROJECT_ROOT, 'data', 'masks')
DATA_RAW   = os.path.join(PROJECT_ROOT, 'data', 'raw')
OUT_YOLO   = os.path.join(PROJECT_ROOT, 'data', 'seg_yolo')
MODELS_DIR = os.path.join(PROJECT_ROOT, 'models'); os.makedirs(MODELS_DIR, exist_ok=True)
SPLIT_MAP  = {'train': 'train', 'val': 'valid', 'test': 'test'}   # OMNI -> nama split YOLO
MIN_AREA   = 8
assert os.path.isdir(DATA_MASKS), 'data/masks belum ada — jalankan notebook 11 dulu.'
print('sumber mask:', os.path.relpath(DATA_MASKS, PROJECT_ROOT))

sumber mask: data/masks


In [2]:
def nama_asli(fn):
    base = fn.split('.rf.')[0]; stem, ext = base.rsplit('_', 1); return f'{stem}.{ext}'
def cari_raw(split, on):
    for cand in [on, on[:-4] + '.jpg', on[:-4] + '.JPG']:
        p = os.path.join(DATA_RAW, split, cand)
        if os.path.exists(p): return p
    return None

def coco_ke_yolo(split):
    jp = glob.glob(os.path.join(DATA_MASKS, split, '**', '_annotations.coco.json'), recursive=True)[0]
    dj = json.load(open(jp))
    tooth_id = [c['id'] for c in dj['categories'] if c['name'] == 'tooth'][0]
    byimg = {}
    for a in dj['annotations']:
        if a['category_id'] == tooth_id: byimg.setdefault(a['image_id'], []).append(a['segmentation'])
    ysplit = SPLIT_MAP[split]
    dimg = os.path.join(OUT_YOLO, ysplit, 'images'); dlab = os.path.join(OUT_YOLO, ysplit, 'labels')
    os.makedirs(dimg, exist_ok=True); os.makedirs(dlab, exist_ok=True)
    ok = miss = 0
    for im in dj['images']:
        segs = byimg.get(im['id']); on = nama_asli(im['file_name']); rawp = cari_raw(split, on)
        if not segs or rawp is None: miss += 1; continue
        H, W = im['height'], im['width']; lines = []
        for s in segs:
            m = coco_mask.decode(s).astype('uint8')
            cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            for c in cnts:
                if cv2.contourArea(c) < MIN_AREA or len(c) < 3: continue
                p = c.reshape(-1, 2).astype(float); p[:, 0] /= W; p[:, 1] /= H
                lines.append('0 ' + ' '.join(f'{x:.5f} {y:.5f}' for x, y in p))
        if not lines: miss += 1; continue
        stem = os.path.splitext(on)[0]
        shutil.copy(rawp, os.path.join(dimg, on))
        open(os.path.join(dlab, stem + '.txt'), 'w').write('\n'.join(lines))
        ok += 1
    return ok, miss

if os.path.isdir(OUT_YOLO): shutil.rmtree(OUT_YOLO)
for split in ['train', 'val', 'test']:
    ok, miss = coco_ke_yolo(split)
    print(f'{split:5s} -> {SPLIT_MAP[split]:5s}: {ok} gambar+label | {miss} dilewati')
print('dataset YOLO:', os.path.relpath(OUT_YOLO, PROJECT_ROOT))

train -> train: 374 gambar+label | 0 dilewati
val   -> valid: 74 gambar+label | 1 dilewati
test  -> test : 106 gambar+label | 0 dilewati
dataset YOLO: data/seg_yolo


In [3]:
# tulis data.yaml
yaml_path = os.path.join(OUT_YOLO, 'data.yaml')
with open(yaml_path, 'w') as f:
    f.write(f"path: {OUT_YOLO}\ntrain: train/images\nval: valid/images\ntest: test/images\n"
            f"names:\n  0: tooth\n")
print(open(yaml_path).read())

path: /Users/rdrusdiati/IOTN-AC/data/seg_yolo
train: train/images
val: valid/images
test: test/images
names:
  0: tooth



In [4]:
# latih YOLOv8-seg
_ensure('ultralytics', 'ultralytics')
from ultralytics import YOLO
EPOCH_SEG, IMGSZ = 60, 512
model = YOLO('yolov8n-seg.pt')          # ganti -s/-m bila mau lebih akurat
model.train(data=yaml_path, epochs=EPOCH_SEG, imgsz=IMGSZ, batch=16,
            project=os.path.join(MODELS_DIR, 'seg_runs'), name='teeth', exist_ok=True)
best = os.path.join(MODELS_DIR, 'seg_runs', 'teeth', 'weights', 'best.pt')
shutil.copy(best, os.path.join(MODELS_DIR, 'teeth_seg_yolov8.pt'))
print('bobot terbaik ->', os.path.relpath(best, PROJECT_ROOT))

New https://pypi.org/project/ultralytics/8.4.115 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.98 🚀 Python-3.13.13 torch-2.8.0 CPU (Apple M5)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/rdrusdiati/IOTN-AC/data/seg_yolo/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, mu

In [5]:
# evaluasi + pratinjau prediksi
import matplotlib.pyplot as plt
seg = YOLO(os.path.join(MODELS_DIR, 'teeth_seg_yolov8.pt'))
metrik = seg.val(data=yaml_path, split='test')
print('mask mAP50:', round(float(metrik.seg.map50), 3), '| mask mAP50-95:', round(float(metrik.seg.map), 3))

contoh = sorted(glob.glob(os.path.join(OUT_YOLO, 'test', 'images', '*')))[:4]
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
for a, p in zip(ax, contoh):
    r = seg.predict(p, verbose=False, conf=0.25)[0]
    a.imshow(r.plot()[..., ::-1]); a.axis('off'); a.set_title(os.path.basename(p)[:12], fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(PROJECT_ROOT, 'outputs', 'seg_preview.png'), dpi=110, bbox_inches='tight'); plt.show()

Ultralytics 8.4.98 🚀 Python-3.13.13 torch-2.8.0 CPU (Apple M5)
YOLOv8n-seg summary (fused): 86 layers, 3,258,259 parameters, 0 gradients, 11.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 83.6±6.6 MB/s, size: 20.1 KB)
val: Scanning /Users/rdrusdiati/IOTN-AC/data/seg_yolo/test/labels... 106 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 106/106 3.0Kit/s 0.0s
val: New cache created: /Users/rdrusdiati/IOTN-AC/data/seg_yolo/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.0s/it 7.1s1.3ss
                   all        106       2341      0.968      0.947      0.986      0.858      0.966      0.944      0.979      0.712
Speed: 0.1ms preprocess, 61.6ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to /Users/rdrusdiati/IOTN-AC/notebooks/runs/segment/val
mask mAP50: 0.979 | mask mAP50-95: 0.712


<Figure size 1600x400 with 4 Axes>

In [6]:
# ekspor Core ML untuk iPhone
seg.export(format='coreml', imgsz=IMGSZ, nms=True)
src = os.path.join(MODELS_DIR, 'teeth_seg_yolov8.mlpackage')
print('Core ML tersimpan di sekitar bobot .pt (cek folder models/). ')
print('Di Swift: jalankan seg model -> gabung semua mask gigi -> background hitam + crop bbox -> feed model AC.')

Ultralytics 8.4.98 🚀 Python-3.13.13 torch-2.8.0 CPU (Apple M5)
WARNING ⚠️ CoreML 'nms=True' is only supported for detect models. Forcing 'nms=False'.

PyTorch: starting from '/Users/rdrusdiati/IOTN-AC/models/teeth_seg_yolov8.pt' with input shape (1, 3, 512, 512) BCHW and output shape(s) ((1, 37, 5376), (1, 32, 128, 128)) (6.4 MB)
requirements: Ultralytics requirement ['numpy>=1.14.5,<=2.3.5'] not found, attempting AutoUpdate...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/5.1 MB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━ 4.5/5.1 MB 28.7 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 28.7 MB/s  0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.6
    Uninstalling numpy-2.4.6:
      Successfully uninstalled numpy-2.4.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
whisperx 3.8.6 requires t

Running MIL default pipeline:   0%|          | 0/92 [00:00<?, ? passes/s]/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/coremltools/converters/mil/mil/passes/defs/preprocess.py:273: UserWarning: Output, '1010', of the source model, has been renamed to 'var_1010' in the Core ML model.
  warnings.warn(msg.format(var.name, new_name))
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/coremltools/converters/mil/mil/passes/defs/preprocess.py:273: UserWarning: Output, '1048', of the source model, has been renamed to 'var_1048' in the Core ML model.
  warnings.warn(msg.format(var.name, new_name))
Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 332.66 passes/s]


CoreML: export success ✅ 8.0s, saved as '/Users/rdrusdiati/IOTN-AC/models/teeth_seg_yolov8.mlpackage' (12.6 MB)

Export complete (8.1s)
Results saved to /Users/rdrusdiati/IOTN-AC/models/teeth_seg_yolov8.mlpackage
Predict:         yolo predict task=segment model=/Users/rdrusdiati/IOTN-AC/models/teeth_seg_yolov8.mlpackage imgsz=512 
Validate:        yolo val task=segment model=/Users/rdrusdiati/IOTN-AC/models/teeth_seg_yolov8.mlpackage imgsz=512 data=/Users/rdrusdiati/IOTN-AC/data/seg_yolo/data.yaml  
Visualize:       https://netron.app
Core ML tersimpan di sekitar bobot .pt (cek folder models/). 
Di Swift: jalankan seg model -> gabung semua mask gigi -> background hitam + crop bbox -> feed model AC.


## Langkah berikutnya

- Bawa `teeth_seg_yolov8.mlpackage` + model AC ke Xcode.
- Di app, pipeline: foto → **seg model (ini)** → union mask → masking + crop (persis notebook 12)
  → resize 224 + normalisasi → **model AC** → skor.
- Bila mau lebih akurat: ganti `yolov8n-seg` → `yolov8s-seg`, naikkan `EPOCH_SEG`/`IMGSZ`.